In [ ]:
# foodDB 중복 ID 감지 및 재할당

이 노트북은 `foodDB1.js`에서 중복 `id`를 찾아 `id` 규칙에 따라 재할당하고, 결과를 파일에 다시 저장합니다.

- 데이터 읽기 및 파싱
- 중복 ID 감지
- ID 규칙 정의
- 중복 ID 재할당
- 업데이트 저장 및 검증
</VSCode.Cell>
<VSCode.Cell language="python">
import json
import re
from pathlib import Path

# 파일 경로 설정
file_path = Path(r"c:\Users\ASUS\02_과제\2026년도\웹프로그래밍\실습 및 과제\단체프로젝트\js\foodDB1.js")

# JS 파일에서 foodDB 배열만 추출하는 함수
js_text = file_path.read_text(encoding="utf-8")
array_match = re.search(r"const\s+foodDB\s*=\s*(\[.*\]);", js_text, re.DOTALL)
if not array_match:
    raise ValueError("foodDB 배열을 찾을 수 없습니다.")

food_db_text = array_match.group(1)

# JSON으로 변환 가능한 문자열로 정제
food_db_json = food_db_text.replace("\r\n", "\n")
food_db_json = re.sub(r",\s*\n\s*\]", "\n]", food_db_json)

# JSON 구문 파싱
food_db = json.loads(food_db_json)
print(f"총 항목 수: {len(food_db)}")

# 중복 ID 감지
id_counts = {}
duplicates = []
for item in food_db:
    item_id = item.get("id")
    id_counts[item_id] = id_counts.get(item_id, 0) + 1

for item_id, count in id_counts.items():
    if count > 1:
        duplicates.append((item_id, count))

print("중복 ID:", duplicates)

# ID 규칙 정의
category_map = {
    "밥": 1,
    "면": 2,
    "고기": 3,
    "국물": 4,
    "해산물": 5,
    "분식": 6,
    "양식": 7,
    "기타": 8,
}

type_map = {
    "한식": 1,
    "중식": 2,
    "일식": 3,
    "양식": 4,
    "기타": 5,
}

# 기본 타입 결정 함수
def resolve_type(tags):
    for tag in tags:
        if tag in type_map:
            return type_map[tag]
    return 5

# 중복 ID 재할당
used_ids = set(id_counts.keys())
next_sequence = {}
updated_items = []

for item in food_db:
    if id_counts[item["id"]] > 1:
        category_prefix = category_map.get(item.get("category"), 8)
        type_prefix = resolve_type(item.get("tags", []))

        key = (category_prefix, type_prefix)
        next_sequence[key] = next_sequence.get(key, 1)

        while True:
            new_id = category_prefix * 1000 + type_prefix * 100 + next_sequence[key]
            next_sequence[key] += 1
            if new_id not in used_ids:
                used_ids.add(new_id)
                break

        print(f"재할당: {item['name']} {item['id']} -> {new_id}")
        item["id"] = new_id
    updated_items.append(item)

# 재검증
new_id_counts = {}
for item in updated_items:
    new_id_counts[item["id"]] = new_id_counts.get(item["id"], 0) + 1

conflicts = [item_id for item_id, count in new_id_counts.items() if count > 1]
print("재검증 중복 ID:", conflicts)

# JS 파일로 저장
new_json_text = json.dumps(updated_items, ensure_ascii=False, indent=2)
new_js_text = js_text[: array_match.start(1)] + new_json_text + js_text[array_match.end(1) :]
file_path.write_text(new_js_text, encoding="utf-8")
print("수정된 foodDB1.js를 저장했습니다.")
